In [3]:
!nvidia-smi

Sat Aug  6 22:46:35 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   40C    P8     9W /  70W |      0MiB / 15109MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

# **Imports**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# **Load Dataset**

In [5]:
!cp /content/drive/MyDrive/deep-learning-howsam/chapter-1-mlp/classification_Example_by_sahar/mobile_price.zip /content

In [6]:
!unzip /content/mobile_price.zip 

Archive:  /content/mobile_price.zip
  inflating: test.csv                
  inflating: train.csv               


In [2]:
train_path = r"D:\Demis2\Data-class\train.csv"
test_path = r"D:\Demis2\Data-class\test.csv"

In [5]:
df = pd.read_csv(train_path)
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [6]:
X = df.drop('price_range', axis=1)
X.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,842,0,2.2,0,1,0,7,0.6,188,2,2,20,756,2549,9,7,19,0,0,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,6,905,1988,2631,17,3,7,1,1,0
2,563,1,0.5,1,2,1,41,0.9,145,5,6,1263,1716,2603,11,2,9,1,1,0
3,615,1,2.5,0,0,0,10,0.8,131,6,9,1216,1786,2769,16,8,11,1,0,0
4,1821,1,1.2,0,13,1,44,0.6,141,2,14,1208,1212,1411,8,2,15,1,1,0


In [7]:
y = df['price_range']
y

0       1
1       2
2       2
3       2
4       1
       ..
1995    0
1996    2
1997    3
1998    0
1999    3
Name: price_range, Length: 2000, dtype: int64

# **Split**

## Method 1

In [8]:
x_train, x_valid, y_train, y_valid = train_test_split(X, y, train_size=0.7, random_state=42)

In [9]:
x_train.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
836,902,1,0.6,1,0,0,63,0.7,122,5,14,364,1360,3654,18,8,15,0,1,1
575,1197,1,0.5,1,9,0,20,0.1,129,5,10,214,1710,2885,19,7,14,1,1,1
557,1519,0,2.1,0,0,0,32,0.7,200,1,10,168,1239,2912,11,10,15,1,1,0
1235,1971,1,0.5,1,0,0,40,0.3,186,7,19,485,922,571,8,7,17,1,1,0
1360,882,0,0.7,1,9,1,28,0.2,151,6,16,248,884,751,19,11,8,1,0,1


## Method 2

# **Preprocess**

## Convert to tensor

### Train

In [10]:
x_train = torch.FloatTensor(x_train.values)
y_train = torch.LongTensor(y_train.values)

C:\Users\microsoft\AppData\Local\Temp\ipykernel_17460\3633808371.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train = torch.LongTensor(y_train.values)


In [13]:
y_train

tensor([3, 2, 2,  ..., 2, 3, 1])

### Validation

In [11]:
x_valid = torch.FloatTensor(x_valid.values)
y_valid = torch.LongTensor(y_valid.values)

## Standardization

In [12]:
x_train.shape

torch.Size([1400, 20])

In [13]:
x_valid.shape

torch.Size([600, 20])

In [14]:
mu = x_train.mean(dim=0)
std = x_train.std(dim=0)

In [15]:
mu, std

(tensor([1.2403e+03, 4.9357e-01, 1.5257e+00, 5.2357e-01, 4.2664e+00, 5.2786e-01,
         3.2241e+01, 5.0671e-01, 1.4064e+02, 4.5664e+00, 9.9000e+00, 6.3942e+02,
         1.2463e+03, 2.1226e+03, 1.2185e+01, 5.6521e+00, 1.0949e+01, 7.6429e-01,
         5.0429e-01, 4.9929e-01]),
 tensor([4.4307e+02, 5.0014e-01, 8.2512e-01, 4.9962e-01, 4.3234e+00, 4.9940e-01,
         1.8257e+01, 2.8797e-01, 3.5326e+01, 2.2958e+00, 6.0214e+00, 4.3995e+02,
         4.3039e+02, 1.0827e+03, 4.2240e+00, 4.3647e+00, 5.4904e+00, 4.2460e-01,
         5.0016e-01, 5.0018e-01]))

In [16]:
x_train = (x_train - mu) / std
x_valid = (x_valid - mu) / std

# **Dataloader**

## Train

In [17]:
train_data = TensorDataset(x_train, y_train)
train_data

In [19]:
train_data.tensors[0]

tensor([[-0.7636,  1.0126, -1.1219,  ..., -1.8000,  0.9911,  1.0011],
        [-0.0978,  1.0126, -1.2431,  ...,  0.5551,  0.9911,  1.0011],
        [ 0.6289, -0.9869,  0.6960,  ...,  0.5551,  0.9911, -0.9982],
        ...,
        [ 1.3286,  1.0126, -1.2431,  ...,  0.5551, -1.0082,  1.0011],
        [ 1.5498, -0.9869, -0.7583,  ..., -1.8000,  0.9911,  1.0011],
        [-1.3662,  1.0126, -1.1219,  ...,  0.5551, -1.0082, -0.9982]])

In [19]:
train_loader = DataLoader(train_data, 100, True)

In [20]:
len(train_loader)

14

In [21]:
train_loader_iter = iter(train_loader)

In [23]:
next(train_loader_iter)

[tensor([[-1.2466,  1.0126, -0.0312,  ...,  0.5551,  0.9911,  1.0011],
         [ 1.5430, -0.9869,  1.3020,  ...,  0.5551, -1.0082,  1.0011],
         [-0.4454, -0.9869,  1.4232,  ...,  0.5551, -1.0082, -0.9982],
         ...,
         [ 0.8546,  1.0126,  0.4536,  ...,  0.5551, -1.0082, -0.9982],
         [-0.8674,  1.0126, -0.5159,  ...,  0.5551, -1.0082,  1.0011],
         [ 0.4687, -0.9869, -1.2431,  ...,  0.5551,  0.9911,  1.0011]]),
 tensor([0, 3, 0, 0, 0, 2, 1, 2, 2, 0, 2, 1, 1, 2, 2, 3, 2, 2, 1, 3, 2, 2, 0, 0,
         1, 2, 1, 0, 0, 1, 2, 2, 2, 0, 3, 0, 1, 1, 0, 1, 2, 2, 3, 2, 1, 1, 0, 1,
         0, 2, 1, 2, 0, 0, 0, 0, 0, 2, 0, 2, 1, 3, 0, 0, 2, 1, 2, 3, 3, 2, 0, 1,
         0, 0, 0, 3, 0, 2, 3, 2, 0, 3, 1, 1, 0, 1, 2, 3, 0, 1, 3, 2, 3, 3, 1, 2,
         2, 1, 0, 0])]

In [22]:
x_batch, y_batch = next(iter(train_loader))

In [25]:
x_batch.shape

torch.Size([100, 20])

## Validation

In [23]:
valid_dataset = TensorDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=200, shuffle=False)

# **Model**

In [24]:
num_feats = 20
num_class = 4
h1 = 64
h2 = 32

model = nn.Sequential(nn.Linear(num_feats, h1),
                      nn.ReLU(),
                      nn.Linear(h1, h2),
                      nn.ReLU(),
                      nn.Linear(h2, num_class))

In [28]:
model

Sequential(
  (0): Linear(in_features=20, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=4, bias=True)
)

In [29]:
yp = model(x_batch)

In [30]:
yp[:2, :]

tensor([[ 0.0106,  0.1127, -0.0769,  0.1715],
        [ 0.0551,  0.2101, -0.1287,  0.1499]], grad_fn=<SliceBackward0>)

In [31]:
torch.tensor([torch.numel(p) for p in model.parameters()]).sum()

tensor(3556)

In [32]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.0754,  0.0821, -0.2084,  ...,  0.1226,  0.0825, -0.0492],
         [ 0.0263,  0.1090, -0.0943,  ...,  0.1580, -0.1928,  0.0245],
         [-0.1944, -0.0860,  0.1836,  ...,  0.0100, -0.0251, -0.1543],
         ...,
         [ 0.1637,  0.1654,  0.0266,  ..., -0.2174,  0.2085,  0.0482],
         [-0.0982, -0.0817, -0.0137,  ..., -0.1344, -0.0801, -0.0801],
         [ 0.1104, -0.1200, -0.0469,  ...,  0.0570, -0.1359, -0.1029]],
        requires_grad=True), Parameter containing:
 tensor([ 0.0708,  0.0090, -0.0477,  0.0767, -0.0266, -0.2172,  0.1520,  0.2196,
          0.0740,  0.1670,  0.1142,  0.1173,  0.2164, -0.1004,  0.0606, -0.0462,
          0.1044,  0.1042,  0.0376,  0.1055, -0.0800, -0.2006, -0.1966, -0.0636,
         -0.1340, -0.1027, -0.2066, -0.2110,  0.1887, -0.0489,  0.1040,  0.1280,
          0.1581, -0.0850, -0.1109,  0.0439,  0.1205, -0.1469,  0.1359, -0.2063,
          0.1127, -0.1537,  0.0296,  0.0941, -0.0921, -0.0569,  0.1769,  0.0727,

# **Loss & Optimizer**

In [25]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# **Device**

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [27]:
device

'cuda'

In [36]:
model = model.to(device)

In [38]:
model[0].weight

Parameter containing:
tensor([[ 0.0754,  0.0821, -0.2084,  ...,  0.1226,  0.0825, -0.0492],
        [ 0.0263,  0.1090, -0.0943,  ...,  0.1580, -0.1928,  0.0245],
        [-0.1944, -0.0860,  0.1836,  ...,  0.0100, -0.0251, -0.1543],
        ...,
        [ 0.1637,  0.1654,  0.0266,  ..., -0.2174,  0.2085,  0.0482],
        [-0.0982, -0.0817, -0.0137,  ..., -0.1344, -0.0801, -0.0801],
        [ 0.1104, -0.1200, -0.0469,  ...,  0.0570, -0.1359, -0.1029]],
       device='cuda:0', requires_grad=True)

# **Utils**

## Loss

In [28]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [29]:
loss_meter = AverageMeter()

In [30]:
loss_meter.update(3.)

In [31]:
loss_meter.avg

3.0

## Torchmetrics

In [44]:
!pip install torchmetrics

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 419 kB 34.1 MB/s 


In [32]:
target = torch.tensor([0, 1, 2, 3])
preds = torch.tensor([0, 2, 1, 3])

In [33]:
from torchmetrics import Accuracy

In [36]:
acc = Accuracy(task="multiclass", num_classes=4)

In [37]:
acc(preds, target)

tensor(0.5000)

# **Train Loooop!**

In [39]:
num_epochs = 400

loss_train_hist = []
loss_valid_hist = []

acc_train_hist = []
acc_valid_hist = []

model = model.to(device)
for epoch in range(num_epochs):
  loss_train = AverageMeter()
  acc_train = Accuracy(task='multiclass' , num_classes=4).to(device)
  for i, (inputs, targets) in enumerate(train_loader):
    inputs = inputs.to(device)
    targets = targets.to(device)

    outputs = model(inputs)
    
    loss = loss_fn(outputs, targets)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    loss_train.update(loss.item())
    acc_train(outputs, targets)

  with torch.no_grad():
    loss_valid = AverageMeter()
    acc_valid = Accuracy().to(device)
    for i, (inputs, targets) in enumerate(valid_loader):
      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)
      loss = loss_fn(outputs, targets)
      
      loss_valid.update(loss.item())
      acc_valid(outputs, targets)

  loss_train_hist.append(loss_train.avg)
  loss_valid_hist.append(loss_valid.avg)

  acc_train_hist.append(acc_train.compute())
  acc_valid_hist.append(acc_valid.compute())

  if epoch % 10 == 0:
    print(f'Epoch {epoch}')
    print(f'Train: Loss = {loss_train.avg:.4}, Acc = {acc_train.compute():.4}')
    print(f'Valid: Loss = {loss_valid.avg:.4}, Acc = {acc_valid.compute():.4}')
    print()

RuntimeError: CUDA error: CUBLAS_STATUS_ARCH_MISMATCH when calling `cublasCreate(handle)`

In [40]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

2.13.0+cu130
13.0
Quadro P2000
(6, 1)


In [ ]:
torch.sum(outputs.argmax(dim=1) == targets)

In [ ]:
targets.shape

# **Plot**

## Loss

In [ ]:
plt.plot(range(num_epochs), loss_train_hist, 'r-', label='Train')
plt.plot(range(num_epochs), loss_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

## Accuracy

In [ ]:
plt.plot(range(num_epochs), acc_train_hist, 'r-', label='Train')
plt.plot(range(num_epochs), acc_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('Acc')
plt.grid(True)
plt.legend()

# **Save model**

In [ ]:
torch.save(model, 'model.pth')

In [ ]:
mymodel = torch.load('model.pth')